# 08 — Hybrid IA3+LoRA (fixed)

`get_peft_model(..., IA3Config).add_adapter("lora", LoraConfig)` raises
`ValueError: Cannot combine adapters with different peft types` — a
`PeftModel` is bound to one tuner type, and `PeftMixedModel` does not
support IA3 (checked against installed `peft` — `IA3` is not in
`peft.tuners.mixed.COMPATIBLE_TUNER_TYPES`). This is why the notebook
never ran.

Fix: LoRA goes through `peft` as usual (query/value + classifier via
`modules_to_save`). IA3 is attached **manually** as forward hooks with
plain `nn.Parameter` scaling vectors on `attention.self.key`,
`intermediate.dense`, and `output.dense` — the last one filtered to
exclude `attention.output.dense`, which the original
`target_modules=["...","output.dense"]` matched too via `peft`'s
`endswith` matching, silently pulling attention's output projection
into what the comments called "strictly feedforward" IA3.

Also fixed: original run never called `save_pretrained` — 108→36
comparable adapters were being trained and discarded, leaving no way
to do held-out test evaluation (notebook 09). This version saves
every run and supports resume.

In [ ]:
# ------------------------------
# 1. Environment Setup
# ------------------------------
!pip uninstall -y torchao -q
!pip install -q peft --no-deps
!pip install -q accelerate

import torch, torch.nn as nn, transformers, datasets, peft
import os, time, math, gc, json
import numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

print("torch:", torch.__version__, "| peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ------------------------------
# 2. Configuration
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 16
SEEDS = [42, 123, 456]

# All 6 budgets kept, as in the original notebook (comment there noted this was
# a deliberate widen from the Phase 2 plan of 100/500/2000 only). Flagging since
# it also means no 3-way LR sweep is run here (fixed dual-LR, not swept) — if you
# want the originally planned 27+27 (3 budgets x 3 LRs x 3 seeds x 2 langs) sweep
# instead, tell me and I'll adapt this loop.
BUDGETS = [50, 100, 500, 1000, 2000, 20000]
LANGUAGES = ["hi", "te"]
METHOD_NAME = "hybrid_ia3_lora"

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
OUTPUT_ROOT = "/kaggle/working"
ADAPTER_ROOT = f"{OUTPUT_ROOT}/adapters/{METHOD_NAME}"
RESULTS_FILE = f"{OUTPUT_ROOT}/hybrid_experiment_results.csv"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# ------------------------------
# 3. Data + base model helpers
# ------------------------------
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).cuda()

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"], truncation=True,
                          padding="max_length", max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels").map(tokenize, batched=True)
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels").map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])
    train_ds.set_format("torch"); valid_ds.set_format("torch")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=32)
    return train_loader, valid_loader

In [ ]:
# ------------------------------
# 4. Manual IA3 scaling vectors (forward hooks)
# ------------------------------
# key                  -> scales the OUTPUT (size hidden_size)      -- classic IA3 attention target
# intermediate.dense   -> scales the INPUT  (size hidden_size)      -- feedforward, pre-activation
# output.dense (FFN only, "attention" excluded) -> scales the INPUT (size intermediate_size)
#
# query/value are left untouched here (owned by LoRA below), so there is no
# parameter-level overlap between the two adapters despite peft's endswith-based
# target_modules matching being what broke the original "strictly feedforward" claim
# (it silently also matched attention.output.dense).

def attach_ia3_vectors(model, hidden_size, intermediate_size):
    vectors = {}
    handles = []

    def make_output_hook(vec):
        def hook(module, inputs, output):
            return output * vec
        return hook

    def make_input_hook(vec):
        def hook(module, inputs):
            return (inputs[0] * vec,)
        return hook

    for name, module in model.named_modules():
        if name.endswith("attention.self.key"):
            vec = nn.Parameter(torch.ones(hidden_size))
            vectors[name] = vec
            handles.append(module.register_forward_hook(make_output_hook(vec)))
        elif name.endswith("intermediate.dense"):
            vec = nn.Parameter(torch.ones(hidden_size))
            vectors[name] = vec
            handles.append(module.register_forward_pre_hook(make_input_hook(vec)))
        elif name.endswith("output.dense") and "attention" not in name:
            vec = nn.Parameter(torch.ones(intermediate_size))
            vectors[name] = vec
            handles.append(module.register_forward_pre_hook(make_input_hook(vec)))

    # dots aren't legal in nn.Module/Parameter names -> remap keys for storage,
    # keep the original module names (plain list, not a Module) for the sanity check.
    safe_dict = nn.ParameterDict({k.replace(".", "__"): v for k, v in vectors.items()})
    model.ia3_vectors = safe_dict
    model.ia3_target_names = list(vectors.keys())
    return handles

In [ ]:
# ------------------------------
# 5. Hybrid model builder
# ------------------------------
def build_hybrid_model():
    base = load_base_model()

    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["query", "value"],
        modules_to_save=["classifier"],
    )
    model = get_peft_model(base, lora_config)  # single peft adapter -> no mixed-type conflict

    handles = attach_ia3_vectors(model, base.config.hidden_size, base.config.intermediate_size)
    model = model.cuda()
    return model, handles

# ------------------------------
# Sanity check before spending GPU hours (same spirit as notebook 04's classifier-head check)
# ------------------------------
_m, _h = build_hybrid_model()

leaked = [n for n in _m.ia3_target_names if n.endswith("attention.output.dense")]
assert not leaked, f"STOP -- IA3 is leaking into attention.output.dense: {leaked}"
print(f"IA3 attached to {len(_m.ia3_target_names)} modules across the encoder; "
      f"0 leaked into attention.output.dense (query/value stay LoRA-only).")

lora_p = sum(p.numel() for n, p in _m.named_parameters() if p.requires_grad and "lora_" in n)
ia3_p = sum(p.numel() for n, p in _m.named_parameters() if p.requires_grad and "ia3_vectors" in n)
head_p = sum(p.numel() for n, p in _m.named_parameters()
             if p.requires_grad and "lora_" not in n and "ia3_vectors" not in n)
print(f"Trainable -- LoRA: {lora_p:,} | IA3: {ia3_p:,} | head: {head_p:,} | total: {count_trainable_params(_m):,}")
assert lora_p > 0 and ia3_p > 0 and head_p > 0, "STOP -- one of the three param groups is empty"
print("Classifier trainable:", any("classifier" in n for n, p in _m.named_parameters() if p.requires_grad))

del _m
for h in _h: h.remove()
torch.cuda.empty_cache()

In [ ]:
# ------------------------------
# 6. Train + evaluate one (language, budget, seed) config
# ------------------------------
# Protocol matched to 05's primary sweep for a fair comparison:
#   - same epoch schedule (10/10/10/5/5/3 for budgets 50/100/500/1000/2000/20000)
#   - final-epoch metrics only, no best-checkpoint rescue (05 has none either --
#     it shows 22% real collapse across lora/dora/ia3, which 06 reports as a
#     finding, not something to engineer away; hybrid should be held to the same
#     standard rather than getting a validation-selection advantage the others didn't)
# Kept: gradient clipping (max_norm=1.0) and IA3 LR=1e-3 (down from an untested 5e-3
# that caused outright divergence) -- neither of these picks a favorable epoch or adds
# training signal, they just stop numerical blowup, so they aren't an unfair edge.
# Note: IA3 LR=1e-3 is still a one-shot choice, not a validated sweep like 04 did for
# lora/dora/ia3 -- a real asymmetry, flagged rather than hidden.
def train_and_evaluate_hybrid(language, budget, seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
    train_loader, valid_loader = build_loaders(language, budget)
    model, handles = build_hybrid_model()

    lora_p, ia3_p, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if "lora_" in n: lora_p.append(p)
        elif "ia3_vectors" in n: ia3_p.append(p)
        else: head_p.append(p)
    all_trainable = lora_p + ia3_p + head_p

    optimizer = torch.optim.AdamW([
        {"params": lora_p, "lr": 1e-4},
        {"params": ia3_p, "lr": 1e-3},
        {"params": head_p, "lr": 1e-4},
    ])

    # matches 05's schedule exactly: 50/100/500 -> 10, 1000/2000 -> 5, 20000 -> 3
    epochs = 3 if budget == 20000 else (5 if budget in [1000, 2000] else 10)

    model.train()
    start_time = time.perf_counter()
    for _ in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            loss = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                         labels=batch["labels"]).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
    train_time = time.perf_counter() - start_time

    # Peak memory from a single no-grad forward pass (matches 06's compute_summary note).
    torch.cuda.reset_peak_memory_stats()
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    peak_mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    res = {
        "method": METHOD_NAME, "language": language, "budget": budget, "seed": seed,
        "accuracy": round(accuracy_score(labels, preds), 6),
        "macro_f1": round(f1_score(labels, preds, average="macro"), 6),
        "trainable_params": count_trainable_params(model),
        "training_time_sec": round(train_time, 2),
        "peak_gpu_memory_gb": round(peak_mem_gb, 4),
        "epochs": epochs,
    }

    run_dir = f"{ADAPTER_ROOT}/{language}/budget{budget}_seed{seed}"
    os.makedirs(run_dir, exist_ok=True)
    model.save_pretrained(run_dir)  # LoRA delta + classifier (modules_to_save), final-epoch weights
    torch.save(model.ia3_vectors.state_dict(), os.path.join(run_dir, "ia3_vectors.pt"))

    for h in handles: h.remove()
    del model
    gc.collect(); torch.cuda.empty_cache()
    return res

In [ ]:
# ------------------------------
# 7. Execute matrix (resumable)
# ------------------------------
completed_keys = set()
if os.path.exists(RESULTS_FILE):
    existing = pd.read_csv(RESULTS_FILE)
    for _, row in existing.iterrows():
        completed_keys.add((row["language"], int(row["budget"]), int(row["seed"])))
    print(f"Found {len(completed_keys)} completed runs. Resuming...")

total_runs = len(LANGUAGES) * len(BUDGETS) * len(SEEDS)
run_count = 0

for lang in LANGUAGES:
    for budget in BUDGETS:
        for seed in SEEDS:
            run_count += 1
            key = (lang, budget, seed)
            run_dir = f"{ADAPTER_ROOT}/{lang}/budget{budget}_seed{seed}"
            adapter_done = os.path.exists(os.path.join(run_dir, "adapter_model.safetensors")) and \
                           os.path.exists(os.path.join(run_dir, "ia3_vectors.pt"))
            if key in completed_keys and adapter_done:
                print(f"[{run_count}/{total_runs}] SKIP (done): {lang} | budget={budget} | seed={seed}")
                continue

            print(f"[{run_count}/{total_runs}] Training Hybrid | {lang} | budget={budget} | seed={seed}")
            try:
                res = train_and_evaluate_hybrid(lang, budget, seed)
                print(f"  -> Acc: {res['accuracy']:.4f} | F1: {res['macro_f1']:.4f} | "
                      f"peak_mem: {res['peak_gpu_memory_gb']:.2f}GB")
                pd.DataFrame([res]).to_csv(RESULTS_FILE, mode="a", header=not os.path.exists(RESULTS_FILE), index=False)
            except Exception as e:
                print(f"  -> ERROR: {e}")
            finally:
                gc.collect(); torch.cuda.empty_cache()

print("Hybrid evaluation complete.")
if os.path.exists(RESULTS_FILE):
    print(f"Total rows saved: {len(pd.read_csv(RESULTS_FILE))}/{total_runs}")